In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV
from skorch import NeuralNetClassifier
import joblib

In [2]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv").values.astype(np.int64).ravel()
y_test = pd.read_csv("csv_files/y_test_encoded.csv").values.astype(np.int64).ravel()

In [3]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

In [4]:
# Define the neural network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(X_train_scaled.shape[1], 64)  # Dynamically set the number of input features
        self.fc2 = nn.Linear(64, 32)  # 64 neurons to 32 neurons
        self.fc3 = nn.Linear(32, 4)   # 32 neurons to 4 output classes
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Hyperparameters for grid search
param_grid = {
    'lr': [0.01, 0.001, 0.0001]
}

# Initialize the skorch neural network classifier
net = NeuralNetClassifier(
    Net,
    criterion=nn.CrossEntropyLoss,
    optimizer=optim.Adam,
    max_epochs=100,  # Static number of epochs
    batch_size=32,  # Static batch size
)

# Perform randomized grid search
random_search = RandomizedSearchCV(net, param_grid, n_iter=10, cv=10, verbose=2)
random_search.fit(X_train_scaled, y_train)


d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 10 folds for each of 3 candidates, totalling 30 fits
  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.9553       0.7619        0.5320  0.1099
      2        0.4542       0.8929        0.3417  0.1101
      3        0.3431       0.9018        0.2737  0.1135
      4        0.2866       0.9196        0.2272  0.1167
      5        0.2558       0.9167        0.2137  0.1050
      6        0.2411       0.9256        0.2047  0.1202
      7        0.2331       0.9256        0.1928  0.0987
      8        0.2256       0.9226        0.1976  0.1267
      9        0.2193       0.9315        0.1830  0.1367
     10        0.2142       0.9405        0.1853  0.1096
     11        0.2097       0.9435        0.1821  0.1078
     12        0.2061       0.9256        0.1798  0.1203
     13        0.2010       0.9405        0.1733  0.1124
     14        0.1963       0.9286        0.1702  0.1172
     15        0.1946      

In [7]:
# Get the best model
best_model = random_search.best_estimator_

print(f"Best model during training: {random_search.best_score_}")

# Save the model's state_dict
torch.save(best_model.module_.state_dict(), 'best_model_state_dict.pth')
joblib.dump(random_search, 'random_search.pkl')


Best model during training: 0.9361595077913863


['random_search.pkl']

In [8]:
# Evaluate on the test set
y_pred = best_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f'Best model accuracy: {accuracy:.4f}')

Best model accuracy: 0.9399
